In [ ]:
# Public-repository path setup.
# Run from anywhere inside the repository, or set CEFTAZIDIME_PROJECT_ROOT.
import os
from pathlib import Path

def _repo_root():
    env = os.environ.get("CEFTAZIDIME_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "README.md").exists() and (candidate / "03_Notebooks").exists():
            return candidate
    return here

def _previous_project_root(project_root):
    env = os.environ.get("GENOME_MIC_AMR_PROJECT_ROOT")
    if env:
        return Path(env).expanduser().resolve()
    return (project_root / "external" / "Genome_MIC_AMR_Emergence").resolve()

PROJECT_ROOT = _repo_root()

#@title Cell 24.1 - Overview, paths, and exact hidden-state visualisation design
# Purpose:
# Show the actual sequence structure hidden by the regional-deviation scores
# used in Notebook 23.
#
# Notebook 24 does NOT reduce each region to one deviation score.
# Instead, it keeps the exact unitig presence/absence profile within every
# supported mapped subwindow.
#
# Main outputs:
# 24A - exact regional sequence-state barcode, pathogens ordered by MIC
# 24B - chromosome position vs MIC, coloured by exact regional state rank
# 24C - pairwise supported-region unitig similarity, pathogens ordered by MIC
# 24D - pairwise supported-region similarity versus absolute MIC difference
#
# Exact regional state:
# Two pathogens have the same state in a subwindow only when their complete
# unitig presence/absence vectors in that subwindow are identical.
#
# State rank is phenotype-independent:
# state 1 = most frequent exact state in that region,
# state 2 = second most frequent, etc.
#
# State numbers are local to each region. State 1 in W01_S03 is not the same
# biological sequence as state 1 in W04_S01.
#
# This is a descriptive visualisation notebook. The 14 mapped regions were
# already selected by earlier phenotype-guided ablation, so these figures
# are exploratory and are not independent confirmation.

from pathlib import Path
import json

import numpy as np
import pandas as pd
from scipy import sparse
import matplotlib.pyplot as plt
from IPython.display import display
PROJECT_ROOT = _repo_root()
RESULTS_TABLE_DIR = PROJECT_ROOT / "05_Results" / "Tables"
RESULTS_FIGURE_DIR = PROJECT_ROOT / "05_Results" / "Figures"
RESULTS_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

UNITIG_DIR = PROJECT_ROOT / "04_Intermediate" / "10_Whole_Chromosome_Unitigs"
UNITIG_MATRIX = UNITIG_DIR / "10_variable_unitig_matrix_176xM.npz"
UNITIG_SAMPLES = UNITIG_DIR / "10_unitig_sample_order.csv"

NB21_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "21_Mapped_Priority_Window_Subwindow_Ablation"
)

NB21_ASSIGNMENT = NB21_DIR / "21_mapped_subwindow_assignment.npz"
NB21_FINAL_RESULTS = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_matched_random_results.csv"
)
NB21_QC = (
    RESULTS_TABLE_DIR
    / "21_mapped_subwindow_final_QC.csv"
)

NB24_DIR = (
    PROJECT_ROOT
    / "04_Intermediate"
    / "24_Exact_Mapped_Sequence_State_Visualization"
)
NB24_DIR.mkdir(parents=True, exist_ok=True)

STATE_MATRIX_FILE = (
    NB24_DIR
    / "24_exact_regional_state_matrix.npz"
)

STATE_SUMMARY_FILE = (
    RESULTS_TABLE_DIR
    / "24_exact_regional_state_summary.csv"
)

COMBINATION_TABLE_FILE = (
    RESULTS_TABLE_DIR
    / "24_exact_regional_combination_table.csv"
)

PAIRWISE_TABLE_FILE = (
    RESULTS_TABLE_DIR
    / "24_pairwise_supported_region_similarity.csv.gz"
)

FIG24A_PNG = (
    RESULTS_FIGURE_DIR
    / "24A_exact_regional_sequence_state_barcode.png"
)
FIG24A_PDF = FIG24A_PNG.with_suffix(".pdf")

FIG24B_PNG = (
    RESULTS_FIGURE_DIR
    / "24B_chromosome_location_vs_MIC_exact_sequence_state.png"
)
FIG24B_PDF = FIG24B_PNG.with_suffix(".pdf")

FIG24C_PNG = (
    RESULTS_FIGURE_DIR
    / "24C_pairwise_supported_region_unitig_similarity_ordered_by_MIC.png"
)
FIG24C_PDF = FIG24C_PNG.with_suffix(".pdf")

FIG24D_PNG = (
    RESULTS_FIGURE_DIR
    / "24D_supported_region_similarity_vs_MIC_difference.png"
)
FIG24D_PDF = FIG24D_PNG.with_suffix(".pdf")

FINAL_QC = (
    RESULTS_TABLE_DIR
    / "24_exact_mapped_sequence_state_visualization_final_QC.csv"
)

COMPLETION_FILE = (
    NB24_DIR
    / "24_EXACT_MAPPED_SEQUENCE_STATE_VISUALIZATION_COMPLETE.json"
)

EXPECTED_PATHOGENS = 176
EXPECTED_UNITIGS = 1_287_844
EXPECTED_SUPPORTED_MAPPED = 14
REFERENCE_LENGTH_BP = 4_641_652

for path in [
    PROJECT_ROOT,
    RESULTS_TABLE_DIR,
    UNITIG_MATRIX,
    UNITIG_SAMPLES,
    NB21_ASSIGNMENT,
    NB21_FINAL_RESULTS,
    NB21_QC,
]:
    assert path.exists(), f"Required input not found: {path}"

print("Notebook 24 - Exact Mapped Sequence-State Visualization")
print("Supported mapped subwindows expected:", EXPECTED_SUPPORTED_MAPPED)
print("Primary goal: show actual regional sequence states rather than deviation scores.")
print("No new association test will be performed.")

print(
    "\nTransition: Cell 24.2 will verify Notebook 21 and load "
    "the 14 fixed supported mapped subwindows."
)


In [ ]:
#@title Cell 24.2 - Verify Notebook 21 and load the 14 fixed mapped subwindows
# Purpose:
# Load the same 14 mapped refinement priorities used in Notebook 23.

nb21_qc = pd.read_csv(NB21_QC)

assert len(nb21_qc) == 1
assert bool(
    nb21_qc.loc[
        0,
        "final_QC_pass",
    ]
)

samples = (
    pd.read_csv(
        UNITIG_SAMPLES
    )
    .sort_values(
        "sample_index"
    )
    .reset_index(
        drop=True
    )
)

assert len(samples) == EXPECTED_PATHOGENS
assert np.array_equal(
    samples[
        "sample_index"
    ].to_numpy(
        dtype=int
    ),
    np.arange(
        EXPECTED_PATHOGENS
    ),
)

y = samples[
    "log2_mic"
].to_numpy(
    dtype=float
)

assert np.isfinite(
    y
).all()

X = sparse.load_npz(
    UNITIG_MATRIX
).tocsc()

assert X.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_UNITIGS,
)

mapped_results = pd.read_csv(
    NB21_FINAL_RESULTS
)

mapped_supported = (
    mapped_results.loc[
        mapped_results[
            "larger_drop_than_matched_random_after_within_parent_BH"
        ].astype(
            bool
        )
    ]
    .copy()
)

assert len(
    mapped_supported
) == EXPECTED_SUPPORTED_MAPPED

mapped_supported[
    "reference_midpoint_bp"
] = (
    mapped_supported[
        "reference_start_0_based"
    ].astype(
        float
    )
    + mapped_supported[
        "reference_end_0_based_exclusive"
    ].astype(
        float
    )
) / 2.0

mapped_supported[
    "reference_midpoint_Mb"
] = (
    mapped_supported[
        "reference_midpoint_bp"
    ]
    / 1_000_000.0
)

mapped_supported = (
    mapped_supported
    .sort_values(
        "reference_midpoint_bp"
    )
    .reset_index(
        drop=True
    )
)

with np.load(
    NB21_ASSIGNMENT
) as archive:
    subwindow_code = np.asarray(
        archive[
            "subwindow_code"
        ],
        dtype=np.int16,
    )

assert subwindow_code.shape == (
    EXPECTED_UNITIGS,
)

print("Notebook 21 QC: PASS")

display(
    mapped_supported[
        [
            "parent_group_name",
            "subwindow_name",
            "reference_start_0_based",
            "reference_end_0_based_exclusive",
            "reference_midpoint_Mb",
            "n_unitigs",
            "observed_drop_after_removal",
            "within_parent_BH_q_value",
        ]
    ]
)

print(
    "\nCell 24.2 complete."
)

print(
    "Transition: Cell 24.3 will derive exact pathogen sequence states "
    "within every supported mapped subwindow."
)


In [ ]:
#@title Cell 24.3 - Derive exact regional sequence states and exact 14-region combinations
# Purpose:
# Keep the full binary unitig profile within each supported mapped region.
#
# Two pathogens receive the same regional state only when their complete
# unitig presence/absence profiles are identical within that region.
#
# State ranks are assigned by frequency:
# 1 = most frequent exact profile, 2 = second most frequent, etc.
# MIC is not used to define the states.

state_matrix = np.zeros(
    (
        EXPECTED_PATHOGENS,
        EXPECTED_SUPPORTED_MAPPED,
    ),
    dtype=np.int16,
)

state_summary_rows = []
region_unitig_indices = []

for region_index, row in mapped_supported.iterrows():
    global_code = int(
        row[
            "global_subwindow_code"
        ]
    )

    unitig_indices = np.flatnonzero(
        subwindow_code
        == global_code
    )

    assert len(
        unitig_indices
    ) == int(
        row[
            "n_unitigs"
        ]
    )

    region_unitig_indices.append(
        unitig_indices
    )

    dense_binary = (
        X[
            :,
            unitig_indices
        ]
        .toarray()
        .astype(
            np.uint8,
            copy=False,
        )
    )

    packed = np.packbits(
        dense_binary,
        axis=1,
        bitorder="little",
    )

    (
        unique_profiles,
        raw_inverse,
        raw_counts,
    ) = np.unique(
        packed,
        axis=0,
        return_inverse=True,
        return_counts=True,
    )

    n_exact_states = len(
        raw_counts
    )

    raw_order = np.lexsort(
        (
            np.arange(
                n_exact_states
            ),
            -raw_counts,
        )
    )

    raw_to_rank = np.empty(
        n_exact_states,
        dtype=np.int16,
    )

    raw_to_rank[
        raw_order
    ] = np.arange(
        1,
        n_exact_states + 1,
        dtype=np.int16,
    )

    ranked_state = raw_to_rank[
        raw_inverse
    ]

    state_matrix[
        :,
        region_index
    ] = ranked_state

    ranked_counts = np.bincount(
        ranked_state,
        minlength=n_exact_states + 1,
    )[1:]

    assert int(
        ranked_counts.sum()
    ) == EXPECTED_PATHOGENS

    state_summary_rows.append(
        {
            "subwindow_name": str(
                row[
                    "subwindow_name"
                ]
            ),
            "reference_midpoint_Mb": float(
                row[
                    "reference_midpoint_Mb"
                ]
            ),
            "n_unitigs": len(
                unitig_indices
            ),
            "n_exact_sequence_states": n_exact_states,
            "most_common_state_count": int(
                ranked_counts.max()
            ),
            "most_common_state_fraction": float(
                ranked_counts.max()
                / EXPECTED_PATHOGENS
            ),
            "singleton_state_count": int(
                np.sum(
                    ranked_counts
                    == 1
                )
            ),
        }
    )

state_summary = pd.DataFrame(
    state_summary_rows
)

# Exact 14-region combination for each pathogen.
combination_tuples = [
    tuple(
        state_matrix[
            sample_index,
            :
        ].tolist()
    )
    for sample_index in range(
        EXPECTED_PATHOGENS
    )
]

combination_series = pd.Series(
    combination_tuples
)

combination_counts = combination_series.value_counts()

combination_rank_map = {
    combination: rank
    for rank, combination in enumerate(
        combination_counts.index.tolist(),
        start=1,
    )
}

combination_id = np.asarray(
    [
        combination_rank_map[
            combination
        ]
        for combination in combination_tuples
    ],
    dtype=np.int16,
)

combination_table = pd.DataFrame(
    {
        "sample_index": np.arange(
            EXPECTED_PATHOGENS,
            dtype=int,
        ),
        "log2_mic": y,
        "exact_14_region_combination_id": combination_id,
        "combination_frequency": [
            int(
                combination_counts[
                    combination
                ]
            )
            for combination in combination_tuples
        ],
        "exact_14_region_state_vector": [
            "|".join(
                map(
                    str,
                    combination
                )
            )
            for combination in combination_tuples
        ],
    }
)

state_summary.to_csv(
    STATE_SUMMARY_FILE,
    index=False,
)

combination_table.to_csv(
    COMBINATION_TABLE_FILE,
    index=False,
)

np.savez_compressed(
    STATE_MATRIX_FILE,
    state_matrix=state_matrix,
    sample_index=np.arange(
        EXPECTED_PATHOGENS,
        dtype=np.int16,
    ),
    log2_mic=y,
    region_names=mapped_supported[
        "subwindow_name"
    ].astype(
        str
    ).to_numpy(
        dtype="U16"
    ),
    region_midpoint_Mb=mapped_supported[
        "reference_midpoint_Mb"
    ].to_numpy(
        dtype=float
    ),
    exact_combination_id=combination_id,
)

print("Exact regional sequence states derived: PASS")

display(
    state_summary
)

print(
    "\nExact 14-region combinations:",
    len(
        combination_counts
    ),
)

print(
    "Combinations occurring in >1 pathogen:",
    int(
        np.sum(
            combination_counts.to_numpy()
            > 1
        )
    ),
)

print(
    "Pathogens belonging to recurrent exact combinations:",
    int(
        np.sum(
            combination_table[
                "combination_frequency"
            ].to_numpy()
            > 1
        )
    ),
)

print(
    "\nCell 24.3 complete."
)

print(
    "Transition: Cell 24.4 will draw the exact-state barcode with "
    "pathogens ordered by MIC."
)


In [ ]:
#@title Cell 24.4 - Draw exact regional sequence-state barcode ordered by MIC
# Purpose:
# Show the hidden exact sequence states across the chromosome.
#
# Rows = individual pathogens ordered by log2 MIC, then sample index.
# Columns = supported mapped chromosomal subwindows.
# Cell value = frequency rank of the exact sequence state in that region.
#
# Equal cell values within one column mean exactly identical unitig profiles.
# Values should not be compared as biological identities between columns.

order = np.lexsort(
    (
        np.arange(
            EXPECTED_PATHOGENS
        ),
        y,
    )
)

ordered_state_matrix = state_matrix[
    order,
    :
]

ordered_mic = y[
    order
]

fig, ax = plt.subplots(
    figsize=(13, 10)
)

image = ax.imshow(
    ordered_state_matrix,
    aspect="auto",
    interpolation="nearest",
)

ax.set_title(
    "Exact mapped regional sequence states across 176 blaTEM-1-only pathogens"
)

ax.set_xlabel(
    "Supported mapped chromosomal subwindow"
)

ax.set_xticks(
    np.arange(
        EXPECTED_SUPPORTED_MAPPED
    )
)

ax.set_xticklabels(
    mapped_supported[
        "subwindow_name"
    ].astype(
        str
    ).tolist(),
    rotation=90,
)

unique_mic_values = np.sort(
    np.unique(
        y
    )
)

mic_tick_positions = []
mic_tick_labels = []

for mic_value in unique_mic_values:
    positions = np.flatnonzero(
        ordered_mic
        == mic_value
    )

    if len(
        positions
    ) > 0:
        mic_tick_positions.append(
            float(
                positions.mean()
            )
        )

        mic_tick_labels.append(
            f"{mic_value:g}"
        )

        ax.axhline(
            float(
                positions[-1]
            )
            + 0.5,
            linewidth=0.5,
            alpha=0.25,
        )

ax.set_yticks(
    mic_tick_positions
)

ax.set_yticklabels(
    mic_tick_labels
)

ax.set_ylabel(
    "log2 ceftazidime MIC\n(pathogens ordered within each MIC)"
)

cbar = fig.colorbar(
    image,
    ax=ax,
)

cbar.set_label(
    "Exact regional state rank\n(1 = most frequent state within that region)"
)

fig.tight_layout()

fig.savefig(
    FIG24A_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG24A_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print("Saved:")
print(FIG24A_PNG)
print(FIG24A_PDF)

print(
    "\nCell 24.4 complete."
)

print(
    "Transition: Cell 24.5 will place these exact regional states directly "
    "on chromosome position versus MIC."
)


In [ ]:
#@title Cell 24.5 - Draw chromosome position versus MIC using exact regional state ranks
# Purpose:
# Preserve the requested axes while showing exact regional states.
#
# X = MG1655 chromosomal location.
# Y = log2 ceftazidime MIC.
# Point value = exact regional sequence-state rank.
#
# A small fixed display-only vertical offset separates pathogens with the
# same MIC. The offset does not change the underlying MIC.

display_y = y.copy()
maximum_offset = 0.22

for mic_value in unique_mic_values:
    sample_indices = np.flatnonzero(
        y
        == mic_value
    )

    sample_indices = np.sort(
        sample_indices
    )

    if len(
        sample_indices
    ) == 1:
        offsets = np.array(
            [0.0]
        )

    else:
        offsets = np.linspace(
            -maximum_offset,
            maximum_offset,
            len(
                sample_indices
            ),
        )

    display_y[
        sample_indices
    ] = (
        mic_value
        + offsets
    )

fig, ax = plt.subplots(
    figsize=(14, 8)
)

for region_index, row in mapped_supported.iterrows():
    x_value = float(
        row[
            "reference_midpoint_Mb"
        ]
    )

    ax.scatter(
        np.full(
            EXPECTED_PATHOGENS,
            x_value,
            dtype=float,
        ),
        display_y,
        c=state_matrix[
            :,
            region_index
        ],
        s=24,
        alpha=0.65,
        linewidths=0,
    )

ax.set_xlim(
    0,
    REFERENCE_LENGTH_BP
    / 1_000_000.0,
)

ax.set_xlabel(
    "MG1655 chromosomal location (Mb)"
)

ax.set_ylabel(
    "log2 ceftazidime MIC"
)

ax.set_title(
    "Exact mapped regional sequence states across ceftazidime MIC"
)

ax.set_yticks(
    unique_mic_values
)

ax.set_xticks(
    mapped_supported[
        "reference_midpoint_Mb"
    ].to_numpy(
        dtype=float
    )
)

ax.set_xticklabels(
    mapped_supported[
        "subwindow_name"
    ].astype(
        str
    ).tolist(),
    rotation=90,
)

for mic_value in unique_mic_values:
    ax.axhline(
        mic_value,
        linewidth=0.5,
        alpha=0.18,
    )

fig.tight_layout()

fig.savefig(
    FIG24B_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG24B_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print("Saved:")
print(FIG24B_PNG)
print(FIG24B_PDF)

print(
    "\nImportant: state ranks are region-specific categorical labels. "
    "Matching colours are meaningful only within the same chromosomal subwindow."
)

print(
    "\nCell 24.5 complete."
)

print(
    "Transition: Cell 24.6 will calculate pairwise similarity using the "
    "actual unitig states across all 14 supported mapped regions together."
)


In [ ]:
#@title Cell 24.6 - Calculate and draw pairwise supported-region unitig similarity
# Purpose:
# Ask whether pathogens with similar MIC also have similar actual sequence
# combinations across the 14 supported mapped regions.
#
# All unitigs from the 14 supported mapped subwindows are combined.
# Pairwise Jaccard similarity is calculated from the original binary unitig
# presence/absence vectors:
#
#   similarity(i,j) = shared-present / present-in-either
#
# This preserves the actual supported-region unitig states rather than using
# the regional-deviation score or exact-state rank.

all_supported_unitig_indices = np.unique(
    np.concatenate(
        region_unitig_indices
    )
)

X_supported = (
    X[
        :,
        all_supported_unitig_indices
    ]
    .tocsr()
    .astype(
        np.int32
    )
)

assert X_supported.shape[
    0
] == EXPECTED_PATHOGENS

intersection = (
    X_supported
    @ X_supported.T
).toarray().astype(
    np.float64
)

row_present = np.asarray(
    X_supported.sum(
        axis=1
    )
).ravel().astype(
    np.float64
)

union = (
    row_present[
        :,
        None
    ]
    + row_present[
        None,
        :
    ]
    - intersection
)

pairwise_similarity = np.divide(
    intersection,
    union,
    out=np.ones_like(
        intersection,
        dtype=np.float64,
    ),
    where=union > 0,
)

assert pairwise_similarity.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_PATHOGENS,
)

assert np.allclose(
    np.diag(
        pairwise_similarity
    ),
    1.0,
)

ordered_similarity = pairwise_similarity[
    np.ix_(
        order,
        order,
    )
]

fig, ax = plt.subplots(
    figsize=(10, 9)
)

image = ax.imshow(
    ordered_similarity,
    aspect="equal",
    interpolation="nearest",
)

ax.set_title(
    "Pairwise sequence similarity across the 14 supported mapped regions"
)

ax.set_xlabel(
    "Pathogens ordered by log2 ceftazidime MIC"
)

ax.set_ylabel(
    "Pathogens ordered by log2 ceftazidime MIC"
)

boundary_positions = []

for mic_value in unique_mic_values:
    positions = np.flatnonzero(
        ordered_mic
        == mic_value
    )

    if len(
        positions
    ) > 0:
        boundary_positions.append(
            float(
                positions[-1]
            )
            + 0.5
        )

for boundary in boundary_positions:
    ax.axhline(
        boundary,
        linewidth=0.4,
        alpha=0.2,
    )
    ax.axvline(
        boundary,
        linewidth=0.4,
        alpha=0.2,
    )

cbar = fig.colorbar(
    image,
    ax=ax,
)

cbar.set_label(
    "Jaccard similarity of supported-region unitig profiles"
)

fig.tight_layout()

fig.savefig(
    FIG24C_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG24C_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print(
    "Supported mapped unitigs used together:",
    f"{len(all_supported_unitig_indices):,}",
)

print("Saved:")
print(FIG24C_PNG)
print(FIG24C_PDF)

print(
    "\nCell 24.6 complete."
)

print(
    "Transition: Cell 24.7 will plot pairwise supported-region sequence "
    "similarity directly against absolute MIC difference."
)


In [ ]:
#@title Cell 24.7 - Plot supported-region sequence similarity versus absolute MIC difference
# Purpose:
# Directly visualise whether pathogens with more similar supported-region
# unitig combinations tend to have more similar MIC values.
#
# Each point is one unique pathogen pair.
# X = absolute difference in log2 ceftazidime MIC.
# Y = Jaccard similarity across all supported mapped-region unitigs.
#
# This remains descriptive. No regression or significance test is added here.

pair_rows = []

for i in range(
    EXPECTED_PATHOGENS
):
    for j in range(
        i + 1,
        EXPECTED_PATHOGENS
    ):
        pair_rows.append(
            {
                "sample_index_i": i,
                "sample_index_j": j,
                "log2_mic_i": float(
                    y[i]
                ),
                "log2_mic_j": float(
                    y[j]
                ),
                "absolute_log2_mic_difference": float(
                    abs(
                        y[i]
                        - y[j]
                    )
                ),
                "supported_region_unitig_jaccard_similarity": float(
                    pairwise_similarity[
                        i,
                        j
                    ]
                ),
                "exact_14_region_state_match_count": int(
                    np.sum(
                        state_matrix[
                            i,
                            :
                        ]
                        == state_matrix[
                            j,
                            :
                        ]
                    )
                ),
            }
        )

pairwise_table = pd.DataFrame(
    pair_rows
)

pairwise_table.to_csv(
    PAIRWISE_TABLE_FILE,
    index=False,
    compression="gzip",
)

fig, ax = plt.subplots(
    figsize=(9, 7)
)

ax.scatter(
    pairwise_table[
        "absolute_log2_mic_difference"
    ].to_numpy(
        dtype=float
    ),
    pairwise_table[
        "supported_region_unitig_jaccard_similarity"
    ].to_numpy(
        dtype=float
    ),
    s=10,
    alpha=0.25,
    linewidths=0,
)

ax.set_xlabel(
    "Absolute difference in log2 ceftazidime MIC"
)

ax.set_ylabel(
    "Jaccard similarity across supported mapped-region unitigs"
)

ax.set_title(
    "Supported-region sequence similarity versus ceftazidime MIC difference"
)

fig.tight_layout()

fig.savefig(
    FIG24D_PNG,
    dpi=300,
    bbox_inches="tight",
)

fig.savefig(
    FIG24D_PDF,
    bbox_inches="tight",
)

plt.show()
plt.close(
    fig
)

print("Unique pathogen pairs:", len(pairwise_table))

print(
    "Median similarity for identical MIC pairs:",
    float(
        pairwise_table.loc[
            pairwise_table[
                "absolute_log2_mic_difference"
            ]
            == 0,
            "supported_region_unitig_jaccard_similarity",
        ].median()
    ),
)

print(
    "Median similarity for non-identical MIC pairs:",
    float(
        pairwise_table.loc[
            pairwise_table[
                "absolute_log2_mic_difference"
            ]
            > 0,
            "supported_region_unitig_jaccard_similarity",
        ].median()
    ),
)

print("Saved:")
print(FIG24D_PNG)
print(FIG24D_PDF)

print(
    "\nCell 24.7 complete."
)

print(
    "Transition: Cell 24.8 will perform final QC and freeze these "
    "exact-state visualisation outputs for interpretation."
)


In [ ]:
#@title Cell 24.8 - Final QC and freeze exact-state visualisation outputs
# Purpose:
# Confirm that the actual regional-state and pairwise-similarity outputs were
# created successfully.
#
# Notebook 24 ends after descriptive visualisation.
# No causal or independent confirmatory inference is made here.

required_outputs = [
    STATE_MATRIX_FILE,
    STATE_SUMMARY_FILE,
    COMBINATION_TABLE_FILE,
    PAIRWISE_TABLE_FILE,
    FIG24A_PNG,
    FIG24A_PDF,
    FIG24B_PNG,
    FIG24B_PDF,
    FIG24C_PNG,
    FIG24C_PDF,
    FIG24D_PNG,
    FIG24D_PDF,
]

for path in required_outputs:
    assert path.exists(), f"Missing expected output: {path}"
    assert path.stat().st_size > 0, f"Empty expected output: {path}"

assert state_matrix.shape == (
    EXPECTED_PATHOGENS,
    EXPECTED_SUPPORTED_MAPPED,
)

assert len(
    combination_table
) == EXPECTED_PATHOGENS

assert len(
    pairwise_table
) == (
    EXPECTED_PATHOGENS
    * (
        EXPECTED_PATHOGENS - 1
    )
    // 2
)

qc = pd.DataFrame(
    [
        {
            "pathogens": EXPECTED_PATHOGENS,
            "supported_mapped_subwindows": EXPECTED_SUPPORTED_MAPPED,
            "supported_mapped_unitigs_used_together": len(
                all_supported_unitig_indices
            ),
            "exact_14_region_combinations": int(
                combination_table[
                    "exact_14_region_combination_id"
                ].nunique()
            ),
            "recurrent_exact_combinations": int(
                (
                    combination_counts
                    > 1
                ).sum()
            ),
            "unique_pathogen_pairs": len(
                pairwise_table
            ),
            "figure_24A_created": FIG24A_PNG.exists(),
            "figure_24B_created": FIG24B_PNG.exists(),
            "figure_24C_created": FIG24C_PNG.exists(),
            "figure_24D_created": FIG24D_PNG.exists(),
            "new_association_test_performed": False,
            "final_QC_pass": True,
        }
    ]
)

qc.to_csv(
    FINAL_QC,
    index=False,
)

completion_payload = {
    "status": "complete",
    "purpose": "exact mapped sequence-state visualisation",
    "pathogens": EXPECTED_PATHOGENS,
    "supported_mapped_subwindows": mapped_supported[
        "subwindow_name"
    ].astype(
        str
    ).tolist(),
    "exact_14_region_combinations": int(
        combination_table[
            "exact_14_region_combination_id"
        ].nunique()
    ),
    "recurrent_exact_combinations": int(
        (
            combination_counts
            > 1
        ).sum()
    ),
    "new_association_test_performed": False,
    "final_QC_pass": True,
}

COMPLETION_FILE.write_text(
    json.dumps(
        completion_payload,
        indent=2,
    ),
    encoding="utf-8",
)

print("Final QC: PASS")

display(
    qc
)

print(
    "\nNotebook 24 ends here."
)

print(
    "For interpretation, review Figures 24A, 24C, and 24D first. "
    "These retain substantially more of the actual sequence structure "
    "than the regional-deviation figures from Notebook 23."
)
